# Ablation Studies

Systematic analysis of model components for both **BiLSTM** and **Transformer**.

In [ ]:
import sys; sys.path.insert(0, '..')
from src import suppress_logs; suppress_logs()

import torch
torch.set_float32_matmul_precision('medium')

from pytorch_lightning import seed_everything
from src.data import GestureDataModule
from src.ablation import AblationRunner, BILSTM_ABLATIONS, TRANSFORMER_ABLATIONS
from src.ablation.config import BILSTM_QUICK, TRANSFORMER_QUICK
from src.visualization import plot_ablation_comparison, print_results_table

## Configuration

In [ ]:
QUICK_MODE = True      # Use fewer configurations (faster)
DATA_PATH = None       # Auto-download from HuggingFace
SEED = 42

seed_everything(SEED)

# Select configs based on mode
if QUICK_MODE:
    CONFIGS = {'bilstm': BILSTM_QUICK, 'transformer': TRANSFORMER_QUICK}
    print('⚡ Quick mode: fewer configurations')
else:
    CONFIGS = {'bilstm': BILSTM_ABLATIONS, 'transformer': TRANSFORMER_ABLATIONS}
    print('Full mode: all configurations')

for name, cfg in CONFIGS.items():
    print(f'{name.upper()}: {cfg.get_num_configurations()} configs')

In [ ]:
dm = GestureDataModule(data_path=DATA_PATH, seed=SEED)

## Run Ablation Studies

In [ ]:
all_results = {}

for name, config in CONFIGS.items():
    print(f'\n{"="*60}\n{name.upper()} Ablation Study\n{"="*60}')
    
    runner = AblationRunner(config, dm, {'accelerator': 'auto'})
    results = runner.run(verbose=True)
    all_results[name] = results

## Results Comparison

In [ ]:
for name, results in all_results.items():
    results.print_summary()

In [ ]:
fig = plot_ablation_comparison(all_results)
from pathlib import Path
Path('../plots').mkdir(exist_ok=True)
fig.savefig('../plots/ablation_comparison.png', dpi=150)

## Detailed Results

In [ ]:
import pandas as pd

for name, results in all_results.items():
    print(f'\n{name.upper()} Results:')
    display(results.to_dataframe().style.background_gradient(subset=['mean_accuracy'], cmap='Greens'))

## Per-Parameter Analysis

In [ ]:
import matplotlib.pyplot as plt

for name, results in all_results.items():
    df = results.to_dataframe()
    params = df['ablation_param'].unique()
    
    fig, axes = plt.subplots(1, len(params), figsize=(6*len(params), 5))
    if len(params) == 1: axes = [axes]
    
    for ax, param in zip(axes, params):
        p = df[df['ablation_param'] == param].sort_values('mean_accuracy', ascending=False)
        colors = ['gold' if i == 0 else 'steelblue' for i in range(len(p))]
        ax.barh(p['ablation_value'].astype(str), p['mean_accuracy'], xerr=p['std_accuracy'], 
                color=colors, capsize=3)
        ax.set_xlabel('Accuracy')
        ax.set_title(f'{param}')
        ax.set_xlim(0, 1.05)
    
    plt.suptitle(f'{name.upper()} Parameter Analysis', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'../plots/{name}_ablation_details.png', dpi=150)

## Save Results

In [ ]:
from pathlib import Path

save_dir = Path('../results/ablation')
save_dir.mkdir(parents=True, exist_ok=True)

for name, results in all_results.items():
    results.save(str(save_dir / name))
    CONFIGS[name].to_yaml(str(save_dir / name / f'{name}_config.yaml'))

print(f'Results saved to {save_dir}/')

## Best Configurations

In [ ]:
print('\n' + '='*60 + '\nBEST CONFIGURATIONS\n' + '='*60)
for name, results in all_results.items():
    best = max(results.results, key=lambda r: r.mean_accuracy)
    print(f'\n{name.upper()}:')
    print(f'  Config: {best.config_name}')
    print(f'  Accuracy: {best.mean_accuracy:.4f} ± {best.std_accuracy:.4f}')
    print(f'  F1: {best.mean_f1:.4f}')